In [0]:
# 1. Configuration (CHANGE THIS FOR EACH TABLE)
table_name = "Dim_Members" # Change to Dim_Providers, Fact_Claims, etc.
source_path = f"/Volumes/hdfc_ergo/dummy_data/raw/{table_name}.csv"

# 2. Read the data (Anti-fragile: Read as pure string, handle messy healthcare CSVs)
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("multiLine", "true") \
    .option("escape", "\"") \
    .load(source_path)

display(df.limit(5))

In [0]:
from pyspark.sql.functions import current_timestamp, col, from_utc_timestamp
from pyspark.sql.types import StringType

# 1. Standardize Column Names to snake_case
new_cols = [c.lower().replace(" ", "_").replace("-", "_") for c in df.columns]
df = df.toDF(*new_cols)

# 2. CRITICAL: Explicitly Cast ALL columns to STRING (HDFC ERGO Bronze Rule)
for column in df.columns:
    df = df.withColumn(column, col(column).cast(StringType()))

# 3. Add Audit Columns (Unity Catalog Compliant)
df = df.withColumn("_ingested_at", from_utc_timestamp(current_timestamp(), "Asia/Kolkata")) \
       .withColumn("_source_file", col("_metadata.file_path"))

# 4. Display to verify transformations
display(df.limit(5))

In [0]:
# Define the fully qualified table name
bronze_table_fqn = f"hdfc_ergo.hdfc_bronze.{table_name.lower()}"

# Write to Bronze schema (overwriteSchema allows us to safely change types to STRING)
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(bronze_table_fqn)

print(f"✅ Successfully wrote {table_name} to {bronze_table_fqn}")